In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import wandb
import torch
import torch.nn as nn
import torch.distributed as dist
import torch.multiprocessing as mp
import torch.nn.functional as F
import tifffile
from torch import optim
from torch.utils.tensorboard import SummaryWriter
from torch.utils.data import DataLoader, Dataset
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.optim.lr_scheduler import _LRScheduler, ReduceLROnPlateau
from torch.utils.data.distributed import DistributedSampler
from torchvision import transforms
from torchvision.transforms.functional import to_pil_image
from pathlib import Path
import time


import os
import utils, losses
import glob
import math
import datasets
import sys
import numpy as np
import matplotlib.pyplot as plt
import ours_mamba as MODELS
from tqdm import tqdm
from natsort import natsorted

import socket
import multiprocessing
import shutil
import ml_collections
import time

In [ ]:
def get_orochi_B_config():
    config = ml_collections.ConfigDict()
    #encoder
    config.img_size = (256, 256) 
    config.patch_size = 4 
    config.pat_merg_rf = 2
    config.in_chans = 2
    config.embed_dim = 128
    config.depths = (4, 4, 4, 4)
    config.drop_rate = 0
    config.drop_path_rate = 0.2
    config.if_convskip = True
    config.out_indices = (0, 1, 2, 3)
    #mamba
    config.ssm_cfg=None
    config.norm_epsilon=1e-5
    config.initializer_cfg=None
    config.fused_add_norm=True
    config.rms_norm=True
    config.residual_in_fp32=True
    config.patch_norm = True
    config.use_checkpoint = False
    #decoder
    config.decoder_bn = False
    config.decoder_depthseparable = True # This means the decoder is light, if set False, the decoder is dense.
    config.decoder_mode = '2d'
    config.decoder_head_chan = 64
    config.head_sparsity = 0.0


    #training
    config.if_resume = False
    config.finetune_mode = 'fuse_unsup' # 'reg', 'fuse', 'SR', 'IR', fuse_unsup
    config.load_modules = {
        # This config is to just finetue decoder.
        'load': ['encoder', 'decoder'],
        'froze': ['encoder'],
        'unfroze_from_froze': ['norm', 'bias'],
        # This config is to finetue all, referring to "full" finetune in paper.
        # 'froze': [],
        # 'unfroze_from_froze': []
    }  
    config.batch_size = 2
    config.lr = 0.0001
    config.weight_decay = 0.01
    config.warmup_ratio = 0.1
    config.warmup_start_factor = 0.01
    config.max_epoch = 301
    config.gpu_ids = [0] # Set for gpu id, usually 0 if you only have one gpu.
    config.num_workers = min(multiprocessing.cpu_count() * 2, 16)
    config.save_steps = 1 
    config.losses = {
        "mse": (nn.MSELoss(), 1.0),
        "ssim": (losses.SSIM2D(),1.0), 
    }

    #path
    config.checkpoint_dir = './Experiment/BSAFusion2D_unsup/model_best.pth.tar'
    config.data_dir = './data_BSAFusion/My_Dataset'  # 修改为预处理数据的路径
    config.save_dir = f'./Experiment/BSAFusion2D_unsup/{time.strftime("%Y-%m-%d-%H-%M-%S", time.localtime())}_tight_decoder'

    config.modularities = "CT-MRI" # Can be "CT-MRI", "PET-MRI", and "SPECT-MRI"

    # Config for wandb if needed
    config.wandb_key = None
    config.wandb_project = "Orochi"


    return config

In [ ]:
def get_orochi_B_config():
    config = ml_collections.ConfigDict()
    #encoder
    config.img_size = (256, 256) 
    config.patch_size = 4 
    config.pat_merg_rf = 2
    config.in_chans = 6
    config.embed_dim = 128
    config.depths = (4, 4, 4, 4)
    config.drop_rate = 0
    config.drop_path_rate = 0.2
    config.if_convskip = True
    config.out_indices = (0, 1, 2, 3)
    #mamba
    config.ssm_cfg=None
    config.norm_epsilon=1e-5
    config.initializer_cfg=None
    config.fused_add_norm=True
    config.rms_norm=True
    config.residual_in_fp32=True
    config.patch_norm = True
    config.use_checkpoint = False
    #decoder
    config.decoder_bn = False
    config.decoder_depthseparable = True # This means the decoder is light, if set False, the decoder is dense.
    config.decoder_mode = '2d'
    config.decoder_head_chan = 64
    config.head_sparsity = 0.0
    config.decoder_out_chan=3


    #training
    config.if_resume = False
    config.finetune_mode = 'fuse_RGB' # 'fuse', 'fuse_unsup', 'fuse_RGB'
    config.load_modules = {
        # This config is to just finetue decoder.
        'load': ['encoder', 'decoder'],
        'froze': ['encoder'],
        'unfroze_from_froze': ['norm', 'bias'],
        # This config is to finetue all, referring to "full" finetune in paper.
        # 'froze': [],
        # 'unfroze_from_froze': []
    }  
    config.exclude_modules = ['encoder.patch_embed',]
    config.batch_size = 2
    config.lr = 0.0001
    config.weight_decay = 0.01
    config.warmup_ratio = 0.1
    config.warmup_start_factor = 0.01
    config.max_epoch = 301
    config.gpu_ids = [0] # Set for gpu id, usually 0 if you only have one gpu.
    config.num_workers = min(multiprocessing.cpu_count() * 2, 16)
    config.save_steps = 1  
    config.losses = {
        "mse": (nn.MSELoss(), 1.0),
        "ssim": (losses.SSIM2D(),1.0), 
    }

    config.modularities = "SPECT-MRI" # Can be "CT-MRI", "PET-MRI", and "SPECT-MRI"

    #path
    config.checkpoint_dir = './Experiment/BSAFusion2D_unsup/model_best.pth.tar'
    config.data_dir = './data_BSAFusion/My_Dataset'  # 修改为预处理数据的路径
    config.save_dir = f'./Experiment/BSAFusion2D_unsup/{time.strftime("%Y-%m-%d-%H-%M-%S", time.localtime())}_tight_decoder'

    
    # Config for wandb if needed
    config.wandb_key = None
    config.wandb_project = "Orochi"

    return config

In [ ]:
config = get_orochi_B_config() 


image_output_path = Path("./Experiment/BSAFusion2D_unsup/inference_output_images")
# Add date and time into image output path
image_output_path = image_output_path / time.strftime("%Y-%m-%d-%H-%M-%S-lightSPECTMRI", time.localtime())
image_output_path.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda')

start_time = time.time()

model = MODELS.Orochi_Finetune(config).to(device)

test_set = datasets.BSAFusionDataset2D(config, is_train=False)
val_loader = DataLoader(test_set,
                    batch_size=1,
                    # sampler=val_sampler,
                    shuffle=False,
                    num_workers=config.num_workers,
                    pin_memory=True)


test_loader = val_loader
print(f'create dataloader time: {time.time() - start_time:.4f} seconds')
print(f'load data time: {time.time() - start_time:.4f} seconds')


if os.path.isfile(config.checkpoint_dir):
    print(f"=> loading checkpoint '{config.checkpoint_dir}'")
    checkpoint = torch.load(config.checkpoint_dir, map_location=device)

    state_dict = checkpoint['state_dict']
    if "module." in list(state_dict.keys())[0] and "module." not in list(model.state_dict().keys())[0]:
        state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}
    elif "module." not in list(state_dict.keys())[0] and "module." in list(model.state_dict().keys())[0]:
        state_dict = {f"module.{k}":v for k, v in state_dict.items()}
    
    model.load_state_dict(state_dict)
    print(f"=> loaded checkpoint '{config.checkpoint_dir}' (epoch {checkpoint['epoch']})")
else:
    print(f"=> no checkpoint found at '{config.checkpoint_dir}'")

model = model.to(device)
MODELS.print_model_details(model)
print(f'create model time: {time.time() - start_time:.4f} seconds')
    

for name, param in model.named_parameters():
    print(name, param.requires_grad)
torch.cuda.empty_cache()
print(f"Cached memory after empty cache: {torch.cuda.memory_reserved() / 1e9:.2f} GB")
model.eval()
all_psnr = []
all_ssim = []

SSIM_calculator = losses.SSIM2D()

with torch.no_grad():
    for batch_idx, (source1, source2, target) in enumerate(tqdm(val_loader)):
        source = torch.cat([source1, source2], dim=1)
        source = source.to(device)
        target = target.to(device)

        logits, aux_loss = model(source, target)
        restored = logits['restored']
        
        for i in range(source.size(0)):
            psnr_val = utils.psnr(target[i].cpu().numpy(), restored[i].cpu().numpy())
            all_psnr.append(psnr_val)
            ssim_val = 1 - SSIM_calculator(target[i].unsqueeze(dim=0), restored[i].unsqueeze(dim=0))
            all_ssim.append(ssim_val.cpu().numpy())
            
            s1_norm = (source[i, 0:3].cpu().numpy())
            s2_norm = (source[i, 3:6].cpu().numpy())
            r_norm = (restored[i, 0:3].cpu().numpy())

            s1_norm = np.clip(s1_norm, 0, 1)
            s2_norm = np.clip(s2_norm, 0, 1)
            r_norm = np.clip(r_norm, 0, 1)

            s1_norm = np.transpose(s1_norm, (1, 2, 0))  # HWC
            s2_norm = np.transpose(s2_norm, (1, 2, 0))  # HWC
            r_norm = np.transpose(r_norm, (1, 2, 0))  # HWC

            s1_pil = to_pil_image(s1_norm, mode='RGB')
            s2_pil = to_pil_image(s2_norm, mode='RGB')
            r_pil = to_pil_image(r_norm, mode='RGB')
            s1_pil.save(os.path.join(image_output_path, f's1_{batch_idx}_{i}.png'))
            s2_pil.save(os.path.join(image_output_path, f's2_{batch_idx}_{i}.png'))
            r_pil.save(os.path.join(image_output_path, f'r_{batch_idx}_{i}.png'))
            
avg_psnr = np.mean(all_psnr)
avg_ssim = np.mean(all_ssim)
print(f'Average PSNR: {avg_psnr:.4f}')
print(f'Average SSIM: {avg_ssim:.4f}')
torch.cuda.empty_cache()


create dataloader time: 0.4782 seconds
load data time: 0.4783 seconds
=> loading checkpoint '/home/zch/Documents/foundation_mamba_biomed/Experiment/BSAFusion2D_unsup/2025-03-13-10-00-14-SPECT-MRI-light/experiments/model_best.pth.tar'
=> loaded checkpoint '/home/zch/Documents/foundation_mamba_biomed/Experiment/BSAFusion2D_unsup/2025-03-13-10-00-14-SPECT-MRI-light/experiments/model_best.pth.tar' (epoch 176)
Orochi_Finetune(
  (encoder): MambaEncoderHeria(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(6, 128, kernel_size=(4, 4), stride=(4, 4))
      (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    )
    (layers): ModuleList(
      (0): BasicLayer(
        (blocks): ModuleList(
          (0): Block(
            (mixer): Mamba2(
              (in_proj): Linear(in_features=128, out_features=772, bias=False)
              (conv1d): Conv1d(512, 512, kernel_size=(4,), stride=(1,), padding=(3,), groups=512)
              (act): SiLU()
              (norm): RMSNorm()
      

100%|██████████| 77/77 [00:03<00:00, 20.52it/s]

Average PSNR: 22.7945
Average SSIM: 0.8746
